# Connect the map to the answers: development round (`docs/design_map_connect.md`)

Gemma 3 12B (bfloat16). Trains a small LoRA (only the layers after the map reaches the answer position) on
spatial questions, then compares the base and the connected model: accuracy on new scenes, new names, farthest /
closest (never trained), general one-word questions, and the map checks (is the map still there, does reflecting or
erasing it change answers, does the model still answer when it cannot look back at M's sentence).

**Before you press Run all:**
1. Upload the new `geomap_colab.zip` to Google Drive at **`MyDrive/geomap/geomap_colab.zip`** (replace the old one).
2. Colab secrets (key icon), notebook access on: **`HF_TOKEN`** (required), `SLACK_WEBHOOK_URL` (optional).
3. Runtime → Change runtime type → **A100** (needed: training runs in bfloat16). About 45–75 minutes. Then **Run all**.

Each round has a tag (cell 5). The adapter and results are saved on Drive; rerunning skips finished steps. The last
cell downloads `connect_dev_<tag>.zip`; send that file back.


In [ ]:
# 1. GPU check
import subprocess, torch, os
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> A100"
GB = torch.cuda.get_device_properties(0).total_memory / 1e9
assert GB > 35, f"Only {GB:.0f} GB of GPU memory: this needs an A100 (training in bfloat16)"
GEMMA, BS_TRAIN, BS_EVAL = "gemma3-12b", 16, 32
print(f"{GB:.0f} GB: {GEMMA}")


In [ ]:
# 2. Code from Drive; results live ON Drive (saved as they are produced, kept across disconnects)
from google.colab import drive
drive.mount("/content/drive")
import os, shutil, zipfile
D = "/content/drive/MyDrive/geomap"
BUNDLE, RES = f"{D}/geomap_colab.zip", f"{D}/results"
assert os.path.exists(BUNDLE), f"Upload geomap_colab.zip to {BUNDLE}"
shutil.rmtree("/content/geomap", ignore_errors=True)
zipfile.ZipFile(BUNDLE).extractall("/content/geomap")
if not os.path.exists(RES):
    shutil.copytree("/content/geomap/results", RES)
assert os.path.exists("/content/geomap/scripts/connect_dev.py"), "Old bundle: upload the new geomap_colab.zip"
shutil.rmtree("/content/geomap/results")
os.symlink(RES, "/content/geomap/results")
os.chdir("/content/geomap")
os.makedirs(f"{RES}/connect_dev", exist_ok=True)
print("rounds on Drive:", sorted(os.listdir(f"{RES}/connect_dev")))


In [ ]:
# 3. Dependencies, PyTorch only (Colab's TensorFlow breaks transformers' imports); bitsandbytes for 8-bit Gemma
!pip -q install "transformers==4.57.3" nnsight accelerate bitsandbytes
import sys
os.environ.update(PYTHONPATH="/content/geomap/src", USE_TF="0", USE_FLAX="0", USE_TORCH="1",
                  TRANSFORMERS_NO_TF="1", TF_CPP_MIN_LOG_LEVEL="3")
sys.path.insert(0, "/content/geomap/src")

In [ ]:
# 4. Secrets: Hugging Face (required) and Slack (optional)
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
from huggingface_hub import whoami
print("Hugging Face:", whoami(token=os.environ["HF_TOKEN"])["name"])
try:
    os.environ["SLACK_WEBHOOK_URL"] = userdata.get("SLACK_WEBHOOK_URL")
except Exception:
    pass
sys.modules.pop("geomap", None)  # a stale or wrong "geomap" import must not shadow the code
sys.path.insert(0, "/content/geomap/src")
from geomap.checkpoint import notify
notify("[Connect dev] Colab run started (or resumed)")
from huggingface_hub import hf_hub_download
hf_hub_download("google/gemma-3-12b-it", "config.json", token=os.environ["HF_TOKEN"]); print("ok google/gemma-3-12b-it")


In [ ]:
# 5. This round's settings (development phase: change freely, every round is logged)
TAG = "r1"
TRAIN_ARGS = "--from-layer 28 --rank 16 --targets all --lr 1e-4 --epochs 1"


In [ ]:
# 6. Train the adapter (skipped if this tag already has one)
!cd /content/geomap && PYTHONPATH=/content/geomap/src:/content/geomap/scripts USE_TF=0 USE_FLAX=0 USE_TORCH=1 TRANSFORMERS_NO_TF=1 python scripts/connect_dev.py train --tag {TAG} --model {GEMMA} --batch-size {BS_TRAIN} {TRAIN_ARGS} 2>&1 | grep -v -i warning


In [ ]:
# 7. Evaluate base (once, cached) and connected model, write the report
!cd /content/geomap && PYTHONPATH=/content/geomap/src:/content/geomap/scripts USE_TF=0 USE_FLAX=0 USE_TORCH=1 TRANSFORMERS_NO_TF=1 python scripts/connect_dev.py eval --tag {TAG} --model {GEMMA} --batch-size {BS_EVAL} 2>&1 | grep -v -i warning


In [ ]:
# 8. One zip of this round (without the adapter weights, which stay on Drive), downloaded here
import glob
Z = f"/content/connect_dev_{TAG}.zip"
with zipfile.ZipFile(Z, "w") as z:
    for f in glob.glob(f"{RES}/connect_dev/{TAG}/*") + glob.glob(f"{RES}/connect_dev/*.json") + [f"{RES}/connect_dev/LOG.md"]:
        if os.path.exists(f) and not f.endswith(".pt"):
            z.write(f, "results/connect_dev/" + os.path.relpath(f, f"{RES}/connect_dev"))
    print("\n".join(z.namelist()))
shutil.copy(Z, f"{D}/connect_dev_{TAG}.zip")
from google.colab import files
files.download(Z)
